# Addendum — Extending the GRD analysis to include 2020

**Group 03 — Miguel Figueroa, Sebastián Bunzli**

This addendum extends `01_grd_severity_analysis.ipynb` to include 2020 as a
pre-pandemic baseline year, alongside the original 2021-2024 range. The
motivation: 2021 (the first year in the main analysis) was already a full
pandemic year, so we cannot tell from that analysis alone whether the 2021
severity anomaly (see main notebook, Section 6) reflects a genuine pandemic
effect or something specific to 2021. Adding 2020 lets us check whether it
resembles 2022-2024 (supporting "2021 was the anomaly") or resembles 2021
(supporting "the pandemic effect was already present in 2020").

This notebook is self-contained (re-defines the pipeline from the main
notebook) so it can be run independently, but should be read together with
`01_grd_severity_analysis.ipynb` for the full picture — it does not repeat
the Five Cs exploration already done there for 2021-2024.

**Before running:** update `DATA_ROOT` below.

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path


## 1. Configuration — now including 2020

In [ ]:
DATA_ROOT = r"C:\Users\sebal\Downloads\BI_T-main"

# Same GRD_PATHS as the main notebook, with 2020 added. Encoding for 2020
# is assumed latin1 (like 2021/2024) but is verified below, not assumed.
GRD_PATHS = {
    2020: (fr"{DATA_ROOT}\GRD\GRD_PUBLICO_2020.txt", "latin1"),
    2021: (fr"{DATA_ROOT}\GRD\GRD_PUBLICO_2021.txt", "latin1"),
    2022: (fr"{DATA_ROOT}\GRD\GRD_PUBLICO_EXTERNO_2022.txt", "utf-16"),
    2023: (fr"{DATA_ROOT}\GRD\GRD_PUBLICO_2023.txt", "utf-16"),
    2024: (fr"{DATA_ROOT}\GRD\GRD_PUBLICO_2024.txt", "latin1"),
}

NEEDED_COLS = ["PREVISION", "IR_29301_PESO", "IR_29301_SEVERIDAD",
               "IR_29301_MORTALIDAD", "TIPO_INGRESO"]


## 2. Data quality check for 2020 (new year only)

Repeats the same checks done for the other years in the main notebook,
applied to 2020: does the assumed encoding actually yield readable column
names, and are there malformed lines?

In [ ]:
path, enc = GRD_PATHS[2020]

with open(path, "r", encoding=enc) as f:
    header = f.readline()
cols = header.strip().split("|")
missing = [c for c in NEEDED_COLS if c not in cols]
print(f"2020 ({enc}): {len(cols)} columns | missing needed columns: {missing if missing else 'none'}")

with open(path, "r", encoding=enc) as f:
    header = f.readline()
    expected_pipes = header.count("|")
    n_bad = sum(1 for line in f if line.count("|") != expected_pipes)
print(f"2020: {n_bad} malformed line(s) out of the file")


**If the check above shows missing columns:** 2020 is not `latin1`
like assumed — update its encoding in `GRD_PATHS` to `"utf-16"` and re-run
this cell before continuing.

## 3. Cleaning pipeline (same as main notebook)

In [ ]:
INSURANCE_MAP = {
    "FONASA INSTITUCIONAL - (MAI) A": "FONASA", "FONASA INSTITUCIONAL - (MAI) B": "FONASA",
    "FONASA INSTITUCIONAL - (MAI) C": "FONASA", "FONASA INSTITUCIONAL - (MAI) D": "FONASA",
    "FONASA LIBRE ELECCIÓN (FMLE_B)": "FONASA", "FONASA LIBRE ELECCIÓN (FMLE_C)": "FONASA",
    "FONASA LIBRE ELECCIÓN (FMLE_D)": "FONASA",
    "ISAPRE": "PRIVATE", "PARTICULAR": "PRIVATE",
    "DIPRECA": "MILITARY", "CAPREDENA": "MILITARY", "CAJA DE PREVISIÓN FFAA (SISA)": "MILITARY",
    "DESCONOCIDO": None,
}


In [ ]:
# Re-verify TIPO_INGRESO categories for 2020 specifically - do not assume
# the same URGENT_LABELS found for 2024 automatically apply.
path, enc = GRD_PATHS[2020]
df_check_2020 = pd.read_csv(
    path, sep="|", encoding=enc, usecols=["TIPO_INGRESO"], dtype=str,
    engine=("python" if enc == "utf-16" else "c"), on_bad_lines="skip",
)
print(df_check_2020["TIPO_INGRESO"].value_counts(dropna=False))


In [ ]:
URGENT_LABELS = [v for v in df_check_2020["TIPO_INGRESO"].dropna().unique()
                  if "URGEN" in v.upper()]
print("Categories treated as 'urgent' (verified against 2020):", URGENT_LABELS)


In [ ]:
def load_and_clean_grd(year, path, encoding):
    """Same cleaning logic as 01_grd_severity_analysis.ipynb, Section 3."""
    is_utf16 = (encoding == "utf-16")
    df = pd.read_csv(
        path, sep="|", encoding=encoding, usecols=NEEDED_COLS, dtype=str,
        engine=("python" if is_utf16 else "c"), on_bad_lines="skip",
    )
    n_raw = len(df)
    n_duplicates = df.duplicated().sum()

    df["PESO_num"] = pd.to_numeric(
        df["IR_29301_PESO"].str.replace(",", ".", regex=False), errors="coerce"
    )
    df["insurance_group"] = df["PREVISION"].map(INSURANCE_MAP)

    no_grouping = (df["IR_29301_SEVERIDAD"] == "0") & (df["IR_29301_MORTALIDAD"] == "0")

    valid = df[df["insurance_group"].notna() & ~no_grouping &
               (df["IR_29301_SEVERIDAD"] != "DESCONOCIDO")].copy()
    valid["high_severity"] = valid["IR_29301_SEVERIDAD"] == "3"
    valid["urgent_admission"] = valid["TIPO_INGRESO"].isin(URGENT_LABELS)

    stats = {
        "year": year, "n_raw": n_raw, "n_duplicate_rows": n_duplicates,
        "n_unknown_insurance": df["insurance_group"].isna().sum(),
        "n_no_clinical_grouping": no_grouping.sum(),
        "n_valid": len(valid),
    }
    return valid, stats


## 4. Recompute KPIs for all 5 years (2020-2024)

In [ ]:
yearly_data = {}
results, stats_log, admission_results = [], [], []

print("Loading all years...")
for year, (path, enc) in GRD_PATHS.items():
    valid, stats = load_and_clean_grd(year, path, enc)
    yearly_data[year] = valid
    stats_log.append(stats)
    print(f"  {year}: {stats['n_valid']:,} valid rows (of {stats['n_raw']:,} raw)")

    kpi_weight = valid.groupby("insurance_group")["PESO_num"].mean()
    kpi_severity = valid.groupby("insurance_group")["high_severity"].mean() * 100
    kpi_urgent = valid.groupby("insurance_group")["urgent_admission"].mean() * 100
    n = valid["insurance_group"].value_counts()

    results.append({
        "year": year,
        "avg_weight_FONASA": kpi_weight.get("FONASA"),
        "avg_weight_PRIVATE": kpi_weight.get("PRIVATE"),
        "pct_high_sev_FONASA": kpi_severity.get("FONASA"),
        "pct_high_sev_PRIVATE": kpi_severity.get("PRIVATE"),
        "severity_gap_pp": kpi_severity.get("FONASA", 0) - kpi_severity.get("PRIVATE", 0),
        "n_FONASA": n.get("FONASA", 0), "n_PRIVATE": n.get("PRIVATE", 0),
    })
    admission_results.append({
        "year": year,
        "pct_urgent_FONASA": kpi_urgent.get("FONASA"),
        "pct_urgent_PRIVATE": kpi_urgent.get("PRIVATE"),
    })

summary_5yr = pd.DataFrame(results).sort_values("year").reset_index(drop=True)
stats_df_5yr = pd.DataFrame(stats_log).sort_values("year").reset_index(drop=True)
admission_summary_5yr = pd.DataFrame(admission_results).sort_values("year").reset_index(drop=True)

print("\n=== KPI 1 & 2 (2020-2024) ===")
print(summary_5yr)
print("\n=== Load & cleaning stats (2020-2024) ===")
print(stats_df_5yr)
print("\n=== KPI 3 (2020-2024) ===")
print(admission_summary_5yr)


## 5. Interpretation — does 2020 resemble 2021 or 2022-2024?

Read the printed `summary_5yr` table above and compare 2020's
`severity_gap_pp` against 2021 (originally -1.8pp) and 2022-2024 (roughly
+4.9 to +5.5pp):

- **If 2020 is also negative/near zero** → supports the idea that the
  pandemic effect (postponed elective surgeries) was already present in
  2020, and 2021 was not a one-off anomaly but a two-year pandemic pattern.
- **If 2020 looks like 2022-2024 (clearly positive gap)** → supports
  treating 2021 specifically as the anomalous year, with 2020 as a valid
  pre-pandemic baseline confirming the "normal" pattern.

*(Write the actual conclusion here once the printed numbers above are
reviewed — do not assume either direction before checking.)*

## 6. Updated visualizations (5-year trend, 2020-2024)

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(summary_5yr["year"], summary_5yr["pct_high_sev_FONASA"], marker="o", label="FONASA")
plt.plot(summary_5yr["year"], summary_5yr["pct_high_sev_PRIVATE"], marker="o", label="PRIVATE")
plt.ylabel("% high-severity cases")
plt.xlabel("Year")
plt.title("Share of high-severity GRD cases by insurance group (2020-2024)")
plt.legend()
plt.tight_layout()
plt.savefig("severity_gap_trend_2020_2024.png", dpi=150)
plt.close()
print("saved severity_gap_trend_2020_2024.png")


In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(admission_summary_5yr["year"], admission_summary_5yr["pct_urgent_FONASA"], marker="o", label="FONASA")
plt.plot(admission_summary_5yr["year"], admission_summary_5yr["pct_urgent_PRIVATE"], marker="o", label="PRIVATE")
plt.ylabel("% admissions via urgent care")
plt.xlabel("Year")
plt.title("Share of urgent (non-elective) admissions by insurance group (2020-2024)")
plt.legend()
plt.tight_layout()
plt.savefig("admission_urgency_trend_2020_2024.png", dpi=150)
plt.close()
print("saved admission_urgency_trend_2020_2024.png")


## 7. Export updated tables

Saved with a `_2020_2024` suffix so they do not overwrite the original
2021-2024 files from the main notebook — both versions are kept for the
report to reference as needed.

In [ ]:
DATA_IN = Path("data/input")
DATA_OUT = Path("data/output")
DATA_IN.mkdir(parents=True, exist_ok=True)
DATA_OUT.mkdir(parents=True, exist_ok=True)

# Only export the new year's raw extract (2021-2024 extracts already exist
# from the main notebook - no need to regenerate them here).
path, enc = GRD_PATHS[2020]
df_raw_2020 = pd.read_csv(
    path, sep="|", encoding=enc, usecols=NEEDED_COLS, dtype=str,
    engine=("python" if enc == "utf-16" else "c"), on_bad_lines="skip",
)
df_raw_2020.to_csv(DATA_IN / "grd_2020_reduced.csv", index=False, encoding="utf-8")
print(f"input 2020: {len(df_raw_2020):,} rows -> data/input/grd_2020_reduced.csv")

yearly_data[2020][["PREVISION", "insurance_group", "PESO_num", "IR_29301_SEVERIDAD",
                    "IR_29301_MORTALIDAD", "TIPO_INGRESO", "high_severity",
                    "urgent_admission"]].to_csv(
    DATA_OUT / "grd_2020_cleaned.csv", index=False, encoding="utf-8"
)

summary_5yr.to_csv(DATA_OUT / "summary_kpis_2020_2024.csv", index=False)
stats_df_5yr.to_csv(DATA_OUT / "load_stats_2020_2024.csv", index=False)
admission_summary_5yr.to_csv(DATA_OUT / "admission_summary_2020_2024.csv", index=False)

print("\nDone.")
